
# Notebook 10 - Recursive Backtest (J+1, J+2, J+3, J+7)

Objectif:
- utiliser le modèle dual entraîné sur `X_train`
- lancer un forecast récursif depuis le début de `X_test`
- mesurer la performance sur les horizons `J+1`, `J+2`, `J+3`, `J+7`


In [ ]:
from pathlib import Path
from collections import deque

import numpy as np
import pandas as pd
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.metrics import mean_absolute_error

from bike_demand_forecasting.utils import get_paths
from bike_demand_forecasting.training import load_feature_table
from bike_demand_forecasting.features import time_train_test_split
from bike_demand_forecasting.inference import load_models_and_meta, predict_dual
from bike_demand_forecasting.metrics import smape, bias, wape

In [2]:
# Config
ARTIFACT_PREFIX = "catboost_station_3segments"  # modèle entraîné sur X_train
FEATURES_FILENAME = "features_3segments.csv"
HORIZON_DAYS_TO_EVAL = [1, 2, 3, 7]

In [5]:
# Load paths, data, model bundle
project_root = Path.cwd().resolve()
if project_root.name == "notebooks":
    project_root = project_root.parent  # remonte à la racine du repo

paths = get_paths(project_root)

bundle = load_models_and_meta(artifact_prefix=ARTIFACT_PREFIX, parents_n=2)
feature_cols = bundle["feature_cols_cb"]

# Keep split aligned with model schema
drop_cols = ["y_station"] if "is_filled_zero" in feature_cols else ["y_station", "is_filled_zero"]

df_feat = load_feature_table(paths, FEATURES_FILENAME)
X_train, y_train, X_test, y_test, cutoff = time_train_test_split(df_feat, drop_cols=drop_cols)

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("cutoff:", cutoff)
print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("model features:", len(feature_cols))

cutoff: 2025-08-11 00:00:00
X_train: (1474032, 31) X_test: (367224, 31)
model features: 29


In [6]:

def parse_lags_and_windows(cols):
    lags = sorted({int(c.split("_")[1]) for c in cols if c.startswith("lag_")})
    windows = sorted({int(c.split("_")[2]) for c in cols if c.startswith("roll_mean_") or c.startswith("roll_std_")})
    return lags, windows


def next_segment_start(ts):
    ts = pd.Timestamp(ts)
    h = int(ts.hour)
    if h == 0:
        return ts.normalize() + pd.Timedelta(hours=6)
    if h == 6:
        return ts.normalize() + pd.Timedelta(hours=16)
    if h == 16:
        return ts.normalize() + pd.Timedelta(days=1)
    raise ValueError(f"Unexpected segment hour={h}; expected 0/6/16")


def recursive_forecast_from_origin(df_full, bundle, start_ts, days):
    feature_cols = bundle["feature_cols_cb"]
    lags, windows = parse_lags_and_windows(feature_cols)
    max_needed = max(lags + windows) if (lags or windows) else 1

    station_ids = sorted(df_full["start_station_id"].astype(int).unique().tolist())

    # Build slots
    steps = days * 3
    slots = []
    ts = pd.Timestamp(start_ts)
    for _ in range(steps):
        slots.append(ts)
        ts = next_segment_start(ts)

    # Holiday set on forecast window
    cal = USFederalHolidayCalendar()
    holiday_days = set(pd.DatetimeIndex(cal.holidays(start=min(slots).normalize(), end=max(slots).normalize())).normalize())

    # Build historical deques strictly before forecast start
    df_hist = df_full.loc[df_full["date"] < start_ts, ["start_station_id", "date", "y_station"]]
    histories = {}
    for sid in station_ids:
        arr = df_hist.loc[df_hist["start_station_id"].astype(int) == sid, "y_station"].to_numpy(dtype=float)
        if len(arr) < max_needed:
            raise ValueError(f"Station {sid}: history={len(arr)} < required={max_needed}")
        histories[sid] = deque(arr.tolist(), maxlen=max_needed + 400)

    pred_parts = []

    for ts in slots:
        dayofweek = int(ts.dayofweek)
        dayofyear = int(ts.dayofyear)
        month = int(ts.month)
        hour = int(ts.hour)
        segment_id = 0 if hour == 0 else 1 if hour == 6 else 2

        common = {
            "date": ts,
            "segment_id": segment_id,
            "year": int(ts.year),
            "month": month,
            "month_num": month,
            "dayofweek": dayofweek,
            "dayofyear": dayofyear,
            "hour": hour,
            "is_weekend": int(dayofweek >= 5),
            "is_holiday": int(ts.normalize() in holiday_days),
            "dayofw_sin": np.sin(2 * np.pi * dayofweek / 7),
            "dayofw_cos": np.cos(2 * np.pi * dayofweek / 7),
            "dayofy_sin": np.sin(2 * np.pi * dayofyear / 365),
            "dayofy_cos": np.cos(2 * np.pi * dayofyear / 365),
            "month_sin": np.sin(2 * np.pi * (month - 1) / 12),
            "month_cos": np.cos(2 * np.pi * (month - 1) / 12),
            "hour_sin": np.sin(2 * np.pi * hour / 24),
            "hour_cos": np.cos(2 * np.pi * hour / 24),
        }

        rows = []
        for sid in station_ids:
            hist = histories[sid]
            row = {"start_station_id": sid, **common}

            for lag in lags:
                row[f"lag_{lag}"] = hist[-lag]

            for win in windows:
                vals = np.array(list(hist)[-win:], dtype=float)
                row[f"roll_mean_{win}"] = float(vals.mean())
                row[f"roll_std_{win}"] = float(vals.std(ddof=1)) if len(vals) > 1 else np.nan

            rows.append(row)

        df_step = pd.DataFrame(rows)
        missing = sorted(set(feature_cols) - set(df_step.columns))
        if missing:
            raise ValueError(f"Missing features at step {ts}: {missing}")

        y_pred_step = predict_dual(df_step, bundle)

        out_step = df_step[["start_station_id", "date", "segment_id"]].copy()
        out_step["y_pred"] = y_pred_step
        pred_parts.append(out_step)

        # recursive update
        for sid, yhat in zip(df_step["start_station_id"].tolist(), y_pred_step.tolist()):
            histories[int(sid)].append(float(yhat))

    return pd.concat(pred_parts, ignore_index=True)


In [7]:

# Origin at the start of X_test
origin_ts = pd.Timestamp(X_test["date"].min())
max_horizon = max(HORIZON_DAYS_TO_EVAL)

# Use full known history up to origin (train + older rows)
df_full = df_feat[["start_station_id", "date", "segment_id", "y_station"]].copy()

pred_df = recursive_forecast_from_origin(df_full=df_full, bundle=bundle, start_ts=origin_ts, days=max_horizon)
pred_df.head()


,start_station_id,date,segment_id,y_pred
0,30200,2025-08-11,0,0.731869
1,30201,2025-08-11,0,1.062964
2,31000,2025-08-11,0,0.298698
3,31001,2025-08-11,0,0.087992
4,31002,2025-08-11,0,0.563548


In [8]:

# Join actual values from X_test horizon
actual_df = pd.DataFrame({
    "start_station_id": X_test["start_station_id"].astype(int).to_numpy(),
    "date": pd.to_datetime(X_test["date"]).to_numpy(),
    "segment_id": X_test["segment_id"].astype(int).to_numpy(),
    "y_true": y_test.to_numpy(),
})

eval_df = pred_df.merge(actual_df, on=["start_station_id", "date", "segment_id"], how="left")

print("pred rows:", len(pred_df), "| matched y_true:", eval_df["y_true"].notna().sum())
eval_df.head()


pred rows: 17976 | matched y_true: 17976


,start_station_id,date,segment_id,y_pred,y_true
0,30200,2025-08-11,0,0.731869,1
1,30201,2025-08-11,0,1.062964,1
2,31000,2025-08-11,0,0.298698,0
3,31001,2025-08-11,0,0.087992,0
4,31002,2025-08-11,0,0.563548,1


In [9]:

# Metrics by horizon day block: J+1, J+2, J+3, J+7
# 3 segments per day -> day d block = indices [3*(d-1), 3*d)

def metrics_row(y_true, y_pred, label):
    return {
        "horizon": label,
        "n": int(len(y_true)),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "smape": float(smape(y_true, y_pred)),
        "bias": float(bias(y_true, y_pred)),
        "wape": float(wape(y_true, y_pred)),
    }

rows = []
all_slots = sorted(eval_df["date"].unique())

for d in HORIZON_DAYS_TO_EVAL:
    start_i = 3 * (d - 1)
    end_i = 3 * d
    day_slots = all_slots[start_i:end_i]

    m = eval_df["date"].isin(day_slots) & eval_df["y_true"].notna()
    y_true_d = eval_df.loc[m, "y_true"]
    y_pred_d = eval_df.loc[m, "y_pred"]

    rows.append(metrics_row(y_true_d, y_pred_d, f"J+{d}"))

metrics_by_day = pd.DataFrame(rows)
metrics_by_day


,horizon,n,mae,smape,bias,wape
0,J+1,2568,1.869660,1.065858,0.521280,0.297514
1,J+2,2568,1.863905,1.062864,0.595299,0.273156
2,J+3,2568,2.695047,1.148976,1.944390,0.479186
3,J+7,2568,2.497299,1.124486,1.395879,0.424257


In [10]:

# Optional: cumulative metrics up to J+d
rows_cum = []
all_slots = sorted(eval_df["date"].unique())

for d in HORIZON_DAYS_TO_EVAL:
    end_i = 3 * d
    slots_cum = all_slots[:end_i]

    m = eval_df["date"].isin(slots_cum) & eval_df["y_true"].notna()
    y_true_c = eval_df.loc[m, "y_true"]
    y_pred_c = eval_df.loc[m, "y_pred"]

    rows_cum.append(metrics_row(y_true_c, y_pred_c, f"CUM_J+{d}"))

metrics_cum = pd.DataFrame(rows_cum)
metrics_cum


,horizon,n,mae,smape,bias,wape
0,CUM_J+1,2568,1.869660,1.065858,0.521280,0.297514
1,CUM_J+2,5136,1.866782,1.064361,0.558289,0.284834
2,CUM_J+3,7704,2.142871,1.092566,1.020323,0.343187
3,CUM_J+7,17976,2.220798,1.086069,1.058470,0.341828
